<a href="https://colab.research.google.com/github/anirbankhan/AI_Model_figo/blob/main/kerasTuner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense


In [2]:
df = pd.read_csv('diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [6]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [7]:
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=8))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='Adam', loss='binary_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
X = df.iloc[:,:-1]
y = df.iloc[:,-1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [9]:
model.fit(X_train, y_train, batch_size=32, epochs=10, validation_data=(X_test, y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 125ms/step - accuracy: 0.6678 - loss: 0.6477 - val_accuracy: 0.6883 - val_loss: 0.6240
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7020 - loss: 0.6070 - val_accuracy: 0.7013 - val_loss: 0.5880
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7215 - loss: 0.5772 - val_accuracy: 0.7468 - val_loss: 0.5619
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7378 - loss: 0.5545 - val_accuracy: 0.7792 - val_loss: 0.5402
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7492 - loss: 0.5373 - val_accuracy: 0.7857 - val_loss: 0.5238
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7638 - loss: 0.5236 - val_accuracy: 0.7792 - val_loss: 0.5147
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.7736 - loss: 0.5124 - val_accuracy: 0.7792 - val_loss: 0.5028
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7736 - loss: 0.5034 - val_accuracy: 0.7792 - 

In [10]:
# using Keras tuner
!pip install keras_tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 4.9 MB/s eta 0:00:00


In [15]:
# using keras tuner
import keras_tuner as kt
def build_model(hp):
  model = Sequential()
  model.add(Dense(32, activation='relu', input_dim=8))
  model.add(Dense(1, activation='sigmoid'))
  optimizer = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop', 'adadelta'])
  model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
  return model




In [19]:
tuner1 = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=5)
tuner1.search(X_train, y_train, epochs=5, validation_data=(X_test,y_test),verbose=2)

Trial 4 Complete [00h 00m 04s]
val_accuracy: 0.7467532753944397

Best val_accuracy So Far: 0.7792207598686218
Total elapsed time: 00h 00m 14s


In [20]:
tuner1.results_summary()

Results summary
Results in ./untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 2 summary
Hyperparameters:
optimizer: rmsprop
Score: 0.7792207598686218

Trial 3 summary
Hyperparameters:
optimizer: adam
Score: 0.7467532753944397

Trial 1 summary
Hyperparameters:
optimizer: sgd
Score: 0.6623376607894897

Trial 0 summary
Hyperparameters:
optimizer: adadelta
Score: 0.4220779240131378


In [23]:
tuner1.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [24]:
model = tuner1.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [26]:
model.fit(X_train, y_train, batch_size=32, epochs=100, initial_epoch=6, validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 70ms/step - accuracy: 0.7524 - loss: 0.5403 - val_accuracy: 0.7662 - val_loss: 0.5288
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7573 - loss: 0.5215 - val_accuracy: 0.7662 - val_loss: 0.5183
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7622 - loss: 0.5094 - val_accuracy: 0.7727 - val_loss: 0.5099
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7671 - loss: 0.4996 - val_accuracy: 0.7662 - val_loss: 0.5040
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7687 - loss: 0.4922 - val_accuracy: 0.7727 - val_loss: 0.4983
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7704 - loss: 0.4856 - val_accuracy: 0.7857 - val_loss: 0.4943
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7638 - loss: 0.4800 - val_accuracy: 0.7792 - val_loss: 0.4895
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7687 - loss: 0.4762 - val_accurac